In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.ticker import LogLocator
import matplotlib.colors as mcolors

import os

base_path = os.path.expanduser("~/Data/astrospikes/Astrospike/10_27_2025/share_astro_results/v1")

In [ ]:
def load_csv_files(base_dir):
    dict_files = {}
    base_path = os.path.expanduser(base_dir) if isinstance(base_dir, str) else base_dir
    
    for root, _, files in os.walk(base_path):
        for file in files:
            if file.endswith('_metrics.csv'):
                csv_path = os.path.join(root, file)
                df = pd.read_csv(csv_path)
                # Get relative path from base_path
                rel_path = os.path.relpath(csv_path, base_path)
                dict_files[rel_path] = df
    
    return dict_files

dict_files = load_csv_files(base_path)
print(f"Loaded {len(dict_files)} CSV files.")

In [ ]:
def plot_speed_score_cdf(df, ax, label, color, linestyle="-", offset=-5):
    # Extract and sort speed scores
    scores = df["speed_score"].to_numpy()
    scores.sort()
    n = len(scores)
    
    # Compute cumulative distribution
    y = np.arange(1, n + 1) / n

    x_plot, y_plot = scores[:offset], y[:offset]

    # Plot on the given axes with specified style
    ax.plot(x_plot, y_plot, linestyle=linestyle, linewidth=2.0, 
            label=label, color=color)


In [ ]:
def parse_model_info(path):
    parts = path.split("/")
    model_type = parts[0]  # 'float', 'qat', or 'deployed' - this is the model type
    
    # Extract architecture from path
    if "lnes" in path.lower():
        architecture = "lnes"
    elif "two_d_histogram" in path:
        architecture = "two_d_histogram"
    elif "event_frame" in path:
        architecture = "event_frame"
    else:
        architecture = "unknown"
    
    return model_type, architecture

def get_line_style(architecture):
    """Get line style for different architectures."""
    styles = {
        "lnes": "-",              # Solid line
        "two_d_histogram": "--",  # Dashed line
        "event_frame": "-.",      # Dash-dot line
        "unknown": ":"            # Dotted line
    }
    return styles.get(architecture, "-")

def create_short_label(rel_path, model_type, architecture):
    parts = rel_path.split("/")
    label = f"{model_type.upper()} - {architecture.replace('_', ' ').title()}"
    
    # Add results identifier if present (results_5, results_8, etc.)
    if "results_5" in rel_path:
        label += " (5)"
    elif "results_8" in rel_path:
        label += " (8)"
    
    return label

def get_color_for_model(model_type, architecture):
    """
    Get color for a model based on model type only.
    Same model type gets same color (no shades for different architectures).
    """
    # Simple color mapping: blue, red, green
    color_map = {
        "float": "blue",
        "qat": "red",
        "deployed": "green",
        "unknown": "gray"
    }
    
    return color_map.get(model_type, "gray")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_title("Akida V2 - Model Comparison", fontsize=14)

def get_model_type_order(model_type):
    order_map = {
        "deployed": 0,
        "qat": 1,
        "float": 2,
        "unknown": 3
    }
    return order_map.get(model_type, 3)

sorted_items = sorted(dict_files.items(), 
                     key=lambda x: get_model_type_order(parse_model_info(x[0])[0]))

for rel_path, df in sorted_items:
    model_type, architecture = parse_model_info(rel_path)
    label = create_short_label(rel_path, model_type, architecture)
    color = get_color_for_model(model_type, architecture)
    linestyle = get_line_style(architecture)
    
    plot_speed_score_cdf(df, ax, label=label, color=color, 
                         linestyle=linestyle, offset=-10)

ax.set_xlabel("SPEED Score", fontsize=12)
ax.set_ylabel("Fraction of Test Set ($\leq s$)", fontsize=12)
ax.set_yscale('logit')
ax.set_ylim(0.001, 0.999)

ticks = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.5, 0.8, 0.9, 0.95, 0.98, 0.99, 0.995, 0.999]
ticks = [t for t in ticks]
ax.yaxis.set_major_locator(mticker.FixedLocator(ticks))
ax.yaxis.set_major_formatter(mticker.FixedFormatter([f'{t:g}' for t in ticks]))

ax.grid(True, which='both', alpha=0.3)
ax.tick_params(axis='both', which='major', labelsize=10)

ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=10)

plt.tight_layout()
plt.show()
